# Step 19.5 — Baseline Training Run

本节准备未来可复现的 baseline run 协议：记录 speed、memory、loss curve，并保存基线数字。标题中的 run 不代表本轮执行授权；本轮只写材料，所有代码单元未执行、无输出。

19.4提交 `c2fccb8895b0e9076beb6a599833bbac7aeb71b3`已独立核心静态验收PASS，修正0/2；19.2与19.3各累计1/2。这些是材料验收，不是学员掌握或运行反馈。

本节初始生成0/2。完成即停在19.5，不进入Step20。

## 1. 基线协议与候选配置

沿用19.1候选：width=64、layers=2、heads=4、block_size=64、batch_size=4、learning_rate=3e-4，max_seq_len=block_size。20次更新、每5次验证、每次4个validation batches仅是协议候选，不是调优结果。

可复现记录需要固定模型/训练/验证seed、数据文件及SHA-256、90/10 split、软件/设备信息和计时边界。所有数字必须来自未来实际运行；未测项目保持null/未测量。

In [ ]:
import hashlib
import json
import math
import platform
import statistics
import time
from datetime import datetime, timezone
import random
from pathlib import Path
from urllib.request import urlopen

import mlx
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

from mlx.utils import tree_flatten, tree_map

## 2. 数据边界与Sampler

先将连续token stream按90/10切分，再将 `train_tokens` 与 `val_tokens` 分别传给sampler。每个样本取T+1个连续token构造X/Y，各为(B,T)；窗口不跨split边界。Y已shift，loss中不再shift。两侧都必须至少有T+1=65个token。

训练使用持久Python RNG；验证每次使用独立固定seed重建RNG，以复用相同抽样窗口，不消耗训练RNG。它是抽样估计，不代表完整遍历validation split。

In [ ]:
SOURCE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
def acquire_tiny_shakespeare(path, source_url=SOURCE_URL):
    path=Path(path)
    if path.is_file(): return path
    path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(source_url, timeout=30) as response: raw=response.read()
    tmp=path.with_suffix('.tmp'); tmp.write_bytes(raw); tmp.replace(path); return path
def load_cached_character_corpus(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f"Cache missing: {path.resolve()}; correct path or explicitly acquire it.")
    text=path.read_text(encoding='utf-8'); chars=sorted(set(text)); stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for i,ch in enumerate(chars)}
    return text, chars, stoi, itos, mx.array([stoi[ch] for ch in text], dtype=mx.int32)
def split_train_validation(stream, fraction=0.90):
    if len(stream.shape)!=1 or stream.shape[0]<2 or not 0<fraction<1: raise ValueError('Expected 1-D stream and 0<fraction<1')
    s=int(stream.shape[0]*fraction)
    if s<=0 or s>=stream.shape[0]: raise ValueError('Empty split')
    return stream[:s],stream[s:],s
def get_batch(data, batch_size, block_size, rng):
    if len(data.shape)!=1 or type(batch_size) is not int or batch_size<1 or type(block_size) is not int or block_size<1: raise ValueError('Invalid sampler inputs')
    n=data.shape[0]
    if n<block_size+1: raise ValueError('Split needs block_size+1 tokens')
    starts=[rng.randrange(0,n-block_size) for _ in range(batch_size)]; xs=[]; ys=[]
    for s in starts:
        chunk=data[s:s+block_size+1]; xs.append(chunk[:-1]); ys.append(chunk[1:])
    return mx.stack(xs,axis=0),mx.stack(ys,axis=0),starts

In [ ]:
# Local bootstrap: set an explicit repository/cache path if auto-location does not fit your layout.
,def find_project_root(start):
,    current = Path(start).resolve()
,    for candidate in [current] + list(current.parents):
,        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
,            return candidate
,    raise FileNotFoundError("Repository root not found. Set PROJECT_ROOT to the repository root.")
,
,PROJECT_ROOT = find_project_root(Path.cwd())
,CACHE_PATH = PROJECT_ROOT / "data" / "tinyshakespeare" / "input.txt"
,ACQUIRE_MISSING_DATA = False  # explicitly enable only if you choose to download the corpus
,if ACQUIRE_MISSING_DATA:
,    acquire_tiny_shakespeare(CACHE_PATH)
,
,text, characters, stoi, itos, token_stream = load_cached_character_corpus(CACHE_PATH)
,vocab_size = len(characters)
,train_tokens, val_tokens, split_index = split_train_validation(token_stream, 0.90)
,required_tokens = baseline_candidate["block_size"] + 1
,for split_name, split_data in (("train", train_tokens), ("validation", val_tokens)):
,    if split_data.shape[0] < required_tokens:
,        raise ValueError(f"{split_name} split has {split_data.shape[0]} tokens; need at least {required_tokens}.")
,mx.eval(train_tokens, val_tokens)
,print("Vocabulary size:", vocab_size)
,print("Train/validation shapes:", train_tokens.shape, val_tokens.shape)
,
data_sha256 = hashlib.sha256(CACHE_PATH.read_bytes()).hexdigest()


## 3. 完整模型与loss

模型定义保持既有15.4母版的CausalMultiHeadAttention、FeedForwardMLP、TransformerBlock与MiniGPT。恢复时应将这四个类的完整定义复制到本节工作副本，不使用SharedLogitModel代替；架构保持Pre-Norm RMSNorm、因果attention、C→4C→C MLP、独立Block、final RMSNorm和untied LM head。

Loss使用raw logits (B,T,V)，flatten为(B*T,V)与(B*T,)，逐位置cross-entropy后mean为标量()。训练调用nn.value_and_grad；验证直接调用loss_fn。

In [ ]:
def loss_fn(model, X, Y):
    # X, Y: (B, T); Y is already shifted by get_batch.
    logits = model(X)  # (B, T, V), raw unnormalized logits
    B, T, V = logits.shape
    flat_logits = mx.reshape(logits, (B * T, V))  # (B*T, V)
    flat_targets = mx.reshape(Y, (B * T,))        # (B*T,)
    token_losses = nn.losses.cross_entropy(
        flat_logits, flat_targets, axis=-1, reduction="none"
    )  # (B*T,)
    return mx.mean(token_losses)  # scalar MLX array, shape ()


In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


## 4. Train/eval、梯度与求值

`model.train()`设置training mode；`model.eval()`等价于`model.train(False)`。它不关闭梯度、不冻结参数、不自动完成前向。梯度由是否调用梯度变换决定，更新由是否调用optimizer决定。

当前MiniGPT没有Dropout、BatchNorm或读取`self.training`的前向分支；现有Embedding/Linear/GELU/RMSNorm不按模式改变公式。因此相同参数与输入下，单纯切换模式不改变本模型的前向数学定义。

`mx.eval`要求选定数组及依赖完成并返回None；它不是模式切换，也不等于关闭梯度。验证helper不接收optimizer、不调用nn.value_and_grad，因此没有参数更新路径。

In [ ]:
def estimate_validation_loss(model, val_tokens, batch_size, block_size, num_batches, seed):
    if type(num_batches) is not int or num_batches<1: raise ValueError('num_batches must be positive int')
    rng=random.Random(seed); was_training=model.training; model.eval()
    try:
        total_loss=0.0; total_positions=0
        for _ in range(num_batches):
            Xv,Yv,_=get_batch(val_tokens,batch_size,block_size,rng)
            loss=loss_fn(model,Xv,Yv) # no gradient transform, no optimizer
            mx.eval(loss); n=Yv.shape[0]*Yv.shape[1]
            total_loss += loss.item()*n; total_positions += n
        return total_loss/total_positions
    finally:
        model.train(was_training)

## 5. 周期性验证与参数版本

第k次训练调用返回的是更新前的 `L_train(θ[k−1]; B[k])`；更新后模型持有θ[k]，Adam状态也前进一步。到验证周期时，所有validation batch都使用同一当前参数θ[k]，得到`L_val(θ[k])`。不能把训练步返回的旧参数loss当成更新后loss。

若第j批mean loss为Lⱼ、监督位置数为nⱼ，合并平均为 `sum(nⱼ Lⱼ)/sum(nⱼ)`。本节固定B/T，因此每批nⱼ=256，等权batch mean的平均与位置加权平均等价。纸面例：两个假设mean为1.2和1.8，平均1.5；这不是模型结果。

In [ ]:
# One matched model/optimizer/compiled-step setup. Definition only; not executed.
model_kwargs = make_model_kwargs(baseline_candidate, vocab_size=vocab_size)
sampler_kwargs = make_sampler_kwargs(baseline_candidate)
adam_kwargs = make_optimizer_kwargs(baseline_candidate)
mx.random.seed(seeds["model"])
model = MiniGPT(**model_kwargs)
model.train()
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
optimizer = optim.Adam(**adam_kwargs)
optimizer.init(model.trainable_parameters())
training_state = [model.state, optimizer.state]
train_rng = random.Random(seeds["train"])
mx.eval(training_state)

def train_graph(X, Y):
    loss_before_update, grads = loss_and_grad_fn(model, X, Y)
    grad_norm = global_grad_norm(grads)
    mx.eval(loss_before_update, grad_norm)
    require_finite(loss_before_update, "training loss")
    require_finite(grad_norm, "gradient norm")
    grads, preclip_norm, clip_scale = clip_grad_tree(grads, baseline_candidate["clip_max_norm"])
    mx.eval(preclip_norm, clip_scale)
    optimizer.update(model, grads)
    return loss_before_update, grad_norm, preclip_norm, clip_scale

compiled_train_step = mx.compile(train_graph, inputs=training_state, outputs=training_state, shapeless=False)
# Setup only: no model construction or training occurred during material preparation.

In [ ]:
def run_training_with_validation(model, optimizer, train_step, training_state,
                                 train_tokens, val_tokens, train_rng, batch_size, block_size,
                                 num_updates, eval_every, val_batches, val_seed):
    if type(num_updates) is not int or num_updates<1 or type(eval_every) is not int or eval_every<1: raise ValueError('Invalid loop plan')
    model.train(); mx.eval(training_state); completed=int(optimizer.state['step'].item())
    initial_val=estimate_validation_loss(model,val_tokens,batch_size,block_size,val_batches,val_seed)
    history=[{'completed_updates':completed,'train_loss_before_update':None,'validation_loss_current':initial_val}]
    for local_update in range(1,num_updates+1):
        model.train(); X,Y,_=get_batch(train_tokens,batch_size,block_size,train_rng)
        train_loss_before_update=train_step(X,Y)
        mx.eval(train_loss_before_update,training_state); completed += 1
        train_value=train_loss_before_update.item(); val_value=None
        if completed%eval_every==0 or local_update==num_updates:
            val_value=estimate_validation_loss(model,val_tokens,batch_size,block_size,val_batches,val_seed)
        history.append({'completed_updates':completed,'train_loss_before_update':train_value,'validation_loss_current':val_value})
    return history

## 6. 编译状态与未来使用顺序

将模型、nn.value_and_grad、Adam、`[model.state, optimizer.state]`原容器引用和compiled callable作为同一匹配setup。训练图只执行forward/loss、梯度与Adam update并返回更新前loss；采样、模式切换、日志和mx.eval留在图外。编译不等于求值，也不保证提速或省内存。

未来work copy操作顺序：恢复本Notebook中的完整MiniGPT类定义；定位真实缓存，必要时主动启用可选获取；执行本地bootstrap；构造model与optimizer并初始化状态；定义训练图和compiled wrapper；最后才执行训练循环。当前没有loss记录、训练结果或验证结果。

累计修正：16.1/16.2各1/2；16.3—16.5、17.1—17.5、18.1—18.5及19.1各0/2；19.2初始0/2。只编写与发布材料，本轮不执行Notebook、训练、验证、测试、采样、测量或语料读写。停在19.2，不进入19.3；后续逐小节至大Step20完成后停止，不进入21。

## 7. 计时、求值边界与内存窗口

**Wall-clock time（墙钟时间）**使用`time.perf_counter()`。每步计时必须包含该步所需的`mx.eval(loss, training_state)`，否则可能只记录到提交/建立依赖的时间。先做一次warm-up并求值，再记录测量窗口；warm-up本身会推进参数和Adam状态，因此必须纳入协议说明。

MLX的active/cache/peak计数器是分配器计数，不是系统RAM或Swap。先保存起点active/cache，再重置peak；每步完成求值后读计数。所有结果只在未来执行时产生。

In [ ]:
def run_baseline_protocol():
    # Warm-up is a real optimizer update, but is excluded from timed samples.
    X0, Y0, _ = get_batch(train_tokens, baseline_candidate["batch_size"], baseline_candidate["block_size"], train_rng)
    warm_loss, _, _, _ = compiled_train_step(X0, Y0)
    mx.eval(warm_loss, training_state)

    active_start = mx.get_active_memory()
    cache_start = mx.get_cache_memory()
    mx.reset_peak_memory()
    step_seconds, train_losses, grad_norms, clip_scales = [], [], [], []
    validation_points = []
    loop_start = time.perf_counter()

    for update_index in range(1, loop_plan["num_updates"] + 1):
        X, Y, _ = get_batch(train_tokens, baseline_candidate["batch_size"], baseline_candidate["block_size"], train_rng)
        t0 = time.perf_counter()
        loss_before_update, grad_norm, preclip_norm, clip_scale = compiled_train_step(X, Y)
        mx.eval(loss_before_update, training_state, grad_norm, preclip_norm, clip_scale)
        elapsed = time.perf_counter() - t0
        step_seconds.append(elapsed)
        train_losses.append(loss_before_update.item())
        grad_norms.append(preclip_norm.item())
        clip_scales.append(clip_scale.item())

        if update_index % loop_plan["eval_every"] == 0 or update_index == loop_plan["num_updates"]:
            val_loss = estimate_validation_loss(
                model, val_tokens, baseline_candidate["batch_size"],
                baseline_candidate["block_size"], loop_plan["val_batches"],
                seeds["validation"],
            )
            validation_points.append({
                "completed_updates": update_index + 1,
                "validation_loss_current_parameters": val_loss,
            })

    measured_loop_seconds = time.perf_counter() - loop_start
    active_end = mx.get_active_memory()
    cache_end = mx.get_cache_memory()
    raw_peak_after_reset = mx.get_peak_memory()

    return {
        "warmup_updates_excluded_from_timing": 1,
        "timed_updates": len(step_seconds),
        "step_wall_seconds_including_batch_and_eval": step_seconds,
        "training_loss_pre_update": train_losses,
        "validation_points": validation_points,
        "measured_loop_wall_seconds_including_validation": measured_loop_seconds,
        "updates_per_second_excluding_validation": (
            len(step_seconds) / sum(step_seconds) if step_seconds and sum(step_seconds) > 0 else None
        ),
        "active_start_bytes": active_start,
        "cache_start_bytes": cache_start,
        "active_end_bytes": active_end,
        "cache_end_bytes": cache_end,
        "raw_peak_after_reset_bytes": raw_peak_after_reset,
        "window_peak_active_bytes": max(active_start, raw_peak_after_reset),
        "active_delta_bytes": active_end - active_start,
        "gradient_norm_preclip": grad_norms,
        "clip_scale": clip_scales,
    }

# Definition only. Do not call during material preparation.

## 8. 参数版本、记录文件与复现边界

训练步返回的loss属于更新前参数`θ[k−1]`；更新完成后模型是`θ[k]`。验证loss在完整更新后使用当前固定参数`θ[k]`计算，不能把训练旧loss误标为更新后loss。

记录JSON包含配置、数据SHA-256、split索引、随机种子、Python/MLX/平台信息、计时定义、loss曲线和MLX内存计数。没有采集到的温度、Swap或系统压力必须标为null/未测量，不得由MLX allocator counters推断。

时间戳文件名避免覆盖先前基线。重现协议不保证不同设备、系统状态或软件版本逐位一致；它提供足够的条件与数字来解释比较。

In [ ]:
def build_baseline_record(metrics):
    return {
        "schema_version": 1,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "config": dict(baseline_candidate) | dict(loop_plan),
        "seeds": dict(seeds),
        "data": {
            "cache_path": str(CACHE_PATH.resolve()),
            "sha256": data_sha256,
            "split_rule": "contiguous 90/10",
            "split_index": int(split_index),
            "train_tokens": int(train_tokens.shape[0]),
            "validation_tokens": int(val_tokens.shape[0]),
            "vocab_size": int(vocab_size),
        },
        "environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "machine": platform.machine(),
            "mlx_version": mlx.__version__,
            "device": str(mx.default_device()),
        },
        "timing_definition": {
            "step_timer": "perf_counter around compiled step plus mx.eval(loss, training_state)",
            "warmup_updates_excluded": 1,
            "updates_per_second_excludes_validation": True,
            "measured_loop_wall_time_includes_validation": True,
        },
        "metrics": metrics,
        "system_observations": {
            "temperature_celsius": None,
            "swap_used_bytes": None,
            "memory_pressure": None,
            "note": "Only populate these from explicit system observations; never infer them from MLX counters."
        },
    }

# Future run cell: this will train and write a JSON record when executed.
metrics = run_baseline_protocol()
record = build_baseline_record(metrics)
results_dir = PROJECT_ROOT / "results" / "baseline"
results_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
record_path = results_dir / f"baseline_{stamp}.json"
with record_path.open("x", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=2)
print("Saved baseline record:", record_path)
# This cell is deliberately unexecuted in the published master.

## 9. 本节范围与状态

本节只准备可复现baseline协议：速度、loss curve、MLX allocator memory和JSON记录。所有训练/验证/计时/内存读取/文件写入代码仅供未来工作副本使用，本轮没有执行。

19.4已独立核心静态验收PASS，修正0/2；19.2与19.3各累计1/2。19.5初始生成0/2。停止在19.5，不进入Step20；后续按请求逐小节直至大Step20全部材料完成后停止，不进入21。